# Stage 2 — ViT5 VietAI checkpoint, fine-tune domain Y tế

Load `VietAI/vit5-base-vietnews-summarization` (đã pretrain trên VietNewsSum) và fine-tune trên data y tế tự crawl.
**Data**: 780 train + 1560 augment (2340) / 94 val / 105 test  


## 1. Setup

In [1]:
import sys, subprocess, importlib.metadata as md

PACKAGES = [
    'transformers==4.46.3',
    'datasets==3.1.0',
    'evaluate==0.4.3',
    'accelerate==1.1.1',
    'rouge-score==0.1.2',
    'bert-score==0.3.13',
    'sentencepiece==0.2.0',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--no-cache-dir', *PACKAGES])

for pkg in ['transformers', 'datasets', 'evaluate', 'accelerate', 'rouge-score', 'bert-score', 'sentencepiece']:
    print(f'{pkg}: {md.version(pkg)}')

transformers: 4.46.3
datasets: 3.1.0
evaluate: 0.4.3
accelerate: 1.1.1
rouge-score: 0.1.2
bert-score: 0.3.13
sentencepiece: 0.2.0


In [2]:
import os, re, json, gc
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, set_seed,
)
import evaluate

set_seed(42)
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

CUDA: True | NVIDIA L4


## 2. Config

Cấu hình tối ưu cho **L4 24GB · bf16**. Đường dẫn Drive được build ở cell mount.

In [3]:
CONFIG = {
    # Model
    'MODEL_NAME': 'VietAI/vit5-base-vietnews-summarization',
    'MAX_INPUT_LEN': 1024,
    'MAX_TARGET_LEN': 384,

    # Training (L4 24GB · bf16)
    'EPOCHS': 6,
    'LR': 3e-5,
    'TRAIN_BATCH': 1,
    'EVAL_BATCH': 2,
    'GRAD_ACCUM': 8,
    'WEIGHT_DECAY': 0.01,
    'WARMUP_RATIO': 0.06,
    'PATIENCE': 2,
    'LABEL_SMOOTH': 0.1,

    # Generation (eval & inference)
    'NUM_BEAMS': 2,
    'NO_REPEAT_NGRAM': 3,
    'LENGTH_PENALTY': 1.0,

    # Reranking (cell 8)
    'NUM_CANDIDATES': 5,
    'DIVERSITY_PENALTY': 0.5,
    'W_NUMBER': 0.55,
    'W_LENGTH': 0.35,
    'W_HALLUCINATION': 0.60,
    'TARGET_RATIO': 0.225,    # 20–25% → midpoint 22.5%
}
CONFIG

{'MODEL_NAME': 'VietAI/vit5-base-vietnews-summarization',
 'MAX_INPUT_LEN': 1024,
 'MAX_TARGET_LEN': 384,
 'EPOCHS': 6,
 'LR': 3e-05,
 'TRAIN_BATCH': 1,
 'EVAL_BATCH': 2,
 'GRAD_ACCUM': 8,
 'WEIGHT_DECAY': 0.01,
 'WARMUP_RATIO': 0.06,
 'PATIENCE': 2,
 'LABEL_SMOOTH': 0.1,
 'NUM_BEAMS': 2,
 'NO_REPEAT_NGRAM': 3,
 'LENGTH_PENALTY': 1.0,
 'NUM_CANDIDATES': 5,
 'DIVERSITY_PENALTY': 0.5,
 'W_NUMBER': 0.55,
 'W_LENGTH': 0.35,
 'W_HALLUCINATION': 0.6,
 'TARGET_RATIO': 0.225}

## 3. Mount Drive & load dataset

Cell dưới tự dò vị trí folder `data/` trong Drive. Cấu trúc cần có:
```
data/
├── train/train_augmented.jsonl   (2340)
├── val/dataset.jsonl             (94)
└── test/dataset.jsonl            (105)
```

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/data')
required_files = [
    DATA_DIR / 'train' / 'train_augmented.jsonl',
    DATA_DIR / 'val' / 'dataset.jsonl',
    DATA_DIR / 'test' / 'dataset.jsonl',
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Thi?u file data:\n' + '\n'.join(missing))

BASE_DIR = Path('/content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint')
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
BEST_DIR = BASE_DIR / 'best_model'
RL_MODEL_DIR = BASE_DIR / 'model_for_rl'
RESULTS_DIR = BASE_DIR / 'results'
for d in (CHECKPOINT_DIR, BEST_DIR, RL_MODEL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR)
print('BASE_DIR:', BASE_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('BEST_DIR      :', BEST_DIR)
print('RL_MODEL_DIR  :', RL_MODEL_DIR)
print('RESULTS_DIR   :', RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/data
BASE_DIR: /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint
CHECKPOINT_DIR: /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/checkpoints
BEST_DIR      : /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/best_model
RL_MODEL_DIR  : /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/model_for_rl
RESULTS_DIR   : /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/results


In [5]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f]

train_raw = load_jsonl(DATA_DIR / 'train' / 'train_augmented.jsonl')
val_raw   = load_jsonl(DATA_DIR / 'val'   / 'dataset.jsonl')
test_raw  = load_jsonl(DATA_DIR / 'test'  / 'dataset.jsonl')

def to_pair(r):
    return {'id': r['id'], 'source': r['source'], 'summary': r['summary']}

ds = DatasetDict({
    'train': Dataset.from_list([to_pair(r) for r in train_raw]),
    'val':   Dataset.from_list([to_pair(r) for r in val_raw]),
    'test':  Dataset.from_list([to_pair(r) for r in test_raw]),
})
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'summary'],
        num_rows: 2340
    })
    val: Dataset({
        features: ['id', 'source', 'summary'],
        num_rows: 94
    })
    test: Dataset({
        features: ['id', 'source', 'summary'],
        num_rows: 105
    })
})


## 4. Checkpoint `VietAI/vit5-base-vietnews-summarization` đã được fine-tune sẵn trên VietNewsSum (~150K cặp tin tức).

## 5. Tokenizer & model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['MODEL_NAME'])
model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['MODEL_NAME'])

model.gradient_checkpointing_enable()
model.config.use_cache = False

print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("gradient_checkpointing:", model.is_gradient_checkpointing)
print("use_cache:", model.config.use_cache)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Params: 226.0M
gradient_checkpointing: True
use_cache: False


In [7]:
def preprocess(batch):
    enc = tokenizer(batch['source'], max_length=CONFIG['MAX_INPUT_LEN'],
                    truncation=True, padding=False)
    lab = tokenizer(text_target=batch['summary'], max_length=CONFIG['MAX_TARGET_LEN'],
                    truncation=True, padding=False)
    enc['labels'] = lab['input_ids']
    return enc

def token_length_report(split_name, rows):
    src_lens = [len(tokenizer(r['source'], truncation=False)['input_ids']) for r in rows]
    tgt_lens = [len(tokenizer(text_target=r['summary'], truncation=False)['input_ids']) for r in rows]
    return {
        'split': split_name,
        'n': len(rows),
        'src_p95': int(np.percentile(src_lens, 95)),
        'src_max': int(max(src_lens)),
        'src_truncated': int(np.sum(np.array(src_lens) > CONFIG['MAX_INPUT_LEN'])),
        'tgt_p95': int(np.percentile(tgt_lens, 95)),
        'tgt_max': int(max(tgt_lens)),
        'tgt_truncated': int(np.sum(np.array(tgt_lens) > CONFIG['MAX_TARGET_LEN'])),
    }

length_audit = [
    token_length_report('train', train_raw),
    token_length_report('val', val_raw),
    token_length_report('test', test_raw),
]
print(json.dumps(length_audit, ensure_ascii=False, indent=2))

tok_ds = ds.map(preprocess, batched=True, remove_columns=['source', 'summary'])
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding='longest', label_pad_token_id=-100)

[
  {
    "split": "train",
    "n": 2340,
    "src_p95": 1652,
    "src_max": 3409,
    "src_truncated": 748,
    "tgt_p95": 351,
    "tgt_max": 636,
    "tgt_truncated": 63
  },
  {
    "split": "val",
    "n": 94,
    "src_p95": 1307,
    "src_max": 1926,
    "src_truncated": 15,
    "tgt_p95": 318,
    "tgt_max": 421,
    "tgt_truncated": 1
  },
  {
    "split": "test",
    "n": 105,
    "src_p95": 1316,
    "src_max": 2713,
    "src_truncated": 27,
    "tgt_p95": 339,
    "tgt_max": 701,
    "tgt_truncated": 4
  }
]


Map:   0%|          | 0/2340 [00:00<?, ? examples/s]

Map:   0%|          | 0/94 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

## 6. Stage 2 — fine-tune

In [ ]:
def rouge_tokenize(text):
    import re
    import unicodedata
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(text)).casefold())


rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    p_text = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    l_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
    r = rouge.compute(tokenizer=rouge_tokenize, rouge_types=["rouge1", "rouge2", "rougeL"], predictions=p_text, references=l_text, use_stemmer=False)
    return {k: round(v * 100, 2) for k, v in r.items()}

In [9]:
args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CONFIG['EPOCHS'],
    learning_rate=CONFIG['LR'],
    per_device_train_batch_size=CONFIG['TRAIN_BATCH'],
    per_device_eval_batch_size=CONFIG['EVAL_BATCH'],
    gradient_accumulation_steps=CONFIG['GRAD_ACCUM'],
    weight_decay=CONFIG['WEIGHT_DECAY'],
    warmup_ratio=CONFIG['WARMUP_RATIO'],
    label_smoothing_factor=CONFIG['LABEL_SMOOTH'],

    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=None,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Keep this False to avoid OOM during validation.
    predict_with_generate=False,

    bf16=True,
    gradient_checkpointing=True,
    optim='adafactor',

    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tok_ds['train'],
    eval_dataset=tok_ds['val'],
    processing_class=tokenizer,
    data_collator=collator,

    # Do not compute ROUGE during training; final test eval still does it later.
    compute_metrics=None,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['PATIENCE'])],
)


In [10]:
train_result = trainer.train()
trainer.save_state()

trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))

trainer.save_model(str(RL_MODEL_DIR))
tokenizer.save_pretrained(str(RL_MODEL_DIR))

with open(RESULTS_DIR / 'train_result.json', 'w', encoding='utf-8') as f:
    json.dump(train_result.metrics, f, ensure_ascii=False, indent=2)

print('Saved checkpoints under:', CHECKPOINT_DIR)
print('Saved best model      :', BEST_DIR)
print('Saved RL model        :', RL_MODEL_DIR)


Epoch,Training Loss,Validation Loss
0,2.272600,1.893997
2,1.895700,1.696211
4,1.792900,1.685137
5,1.787300,1.675625


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Saved checkpoints under: /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/checkpoints
Saved best model      : /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/best_model
Saved RL model        : /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/model_for_rl


## 7. Evaluation SFT trên test

ROUGE-1/2/L · BERTScore · `number_recall` · `number_hallucination` · `length_compliance` (20–25%).

In [11]:
NUM_RE = re.compile(r'\d+(?:[.,]\d+)*')

def extract_numbers(text):
    return set(NUM_RE.findall(text or ''))

def number_recall(src, pred):
    s = extract_numbers(src)
    if not s: return 1.0
    return len(s & extract_numbers(pred)) / len(s)

def number_hallucination(src, pred):
    p = extract_numbers(pred)
    if not p: return 0.0
    return len(p - extract_numbers(src)) / len(p)

def length_ratio(src, pred):
    sw = max(len(src.split()), 1)
    return len(pred.split()) / sw

def length_compliance(src, pred, lo=0.20, hi=0.25):
    return float(lo <= length_ratio(src, pred) <= hi)

In [12]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_eval_model(model_name_or_path):
    eval_model = AutoModelForSeq2SeqLM.from_pretrained(str(model_name_or_path)).to(DEVICE)
    eval_model.eval()
    return eval_model

def generate_summaries(texts, model_obj=None, batch_size=16, **gen_kwargs):
    eval_model = model_obj if model_obj is not None else model
    eval_model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i+batch_size], max_length=CONFIG['MAX_INPUT_LEN'],
                          truncation=True, padding=True, return_tensors='pt').to(eval_model.device)
        with torch.no_grad():
            ids = eval_model.generate(
                **batch,
                max_length=CONFIG['MAX_TARGET_LEN'],
                num_beams=CONFIG['NUM_BEAMS'],
                no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
                length_penalty=CONFIG['LENGTH_PENALTY'],
                early_stopping=True,
                **gen_kwargs,
            )
        out.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return out

test_src  = ds['test']['source']
test_gold = ds['test']['summary']

# Free training objects before loading eval checkpoints to avoid Colab GPU OOM.
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass
clear_cuda()

base_model = load_eval_model(CONFIG['MODEL_NAME'])
preds_base = generate_summaries(test_src, model_obj=base_model, batch_size=CONFIG['EVAL_BATCH'])
del base_model
clear_cuda()

model = load_eval_model(BEST_DIR)
preds_sft = generate_summaries(test_src, model_obj=model, batch_size=CONFIG['EVAL_BATCH'])
print('Base sample:\n', preds_base[0][:300])
print('\nSFT sample:\n', preds_sft[0][:300])

Base sample:
 Nhiều người tiêm phòng bệnh truyền nhiễm nhưng bệnh lại tái phát , dẫn đến không kịp trở tay .

SFT sample:
 Tư lúc em gái qua đơi do viêm phôi năng, bà Phương , 70 tuôi, ghi chú lịch tiêm phòng hàng năm đê chông ôm. Cụ bà ở phường Gia Định, TP HCM, giải thích gia đình bà từng coi bệnh hô hấp là bệnh nhẹ, chỉ ho vài ba ngày là khỏi. Em gái bà 66 tuổi, từng bị đột quỵ nên chỉ ở nhà, không đi giao tiếp bên n


In [ ]:
def rouge_tokenize(text):
    import re
    import unicodedata
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(text)).casefold())


from bert_score import score as bertscore

def evaluate_predictions(srcs, golds, preds, tag='SFT'):
    r = rouge.compute(tokenizer=rouge_tokenize, rouge_types=["rouge1", "rouge2", "rougeL"], predictions=preds, references=golds, use_stemmer=False)
    P, R, F = bertscore(preds, golds, lang='vi', model_type='xlm-roberta-base', verbose=False)
    nr = [number_recall(s, p)         for s, p in zip(srcs, preds)]
    nh = [number_hallucination(s, p)  for s, p in zip(srcs, preds)]
    lc = [length_compliance(s, p)     for s, p in zip(srcs, preds)]
    lr = [length_ratio(s, p)          for s, p in zip(srcs, preds)]
    return {
        'tag': tag,
        'rouge1':  round(r['rouge1'] * 100, 2),
        'rouge2':  round(r['rouge2'] * 100, 2),
        'rougeL':  round(r['rougeL'] * 100, 2),
        'bertscore_f1':       round(F.mean().item() * 100, 2),
        'number_recall':      round(np.mean(nr) * 100, 2),
        'number_hallucination': round(np.mean(nh) * 100, 2),
        'length_compliance':  round(np.mean(lc) * 100, 2),
        'length_ratio_mean':  round(np.mean(lr), 4),
    }

metrics_base = evaluate_predictions(test_src, test_gold, preds_base, tag='Checkpoint')
metrics_sft = evaluate_predictions(test_src, test_gold, preds_sft, tag='SFT')
metrics_base, metrics_sft

In [14]:
sft_records = [
    {'id': ds['test']['id'][i], 'source': test_src[i],
     'gold': test_gold[i], 'pred_base': preds_base[i], 'pred_sft': preds_sft[i]}
    for i in range(len(test_src))
]
with open(RESULTS_DIR / 'eval_sft.json', 'w', encoding='utf-8') as f:
    json.dump({'metrics': {'checkpoint': metrics_base, 'sft': metrics_sft}, 'predictions': sft_records},
              f, ensure_ascii=False, indent=2)
print('Saved eval_sft.json:', RESULTS_DIR / 'eval_sft.json')

Saved eval_sft.json: /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/results/eval_sft.json


## 8. Candidate generation & reranking

Sinh `K` ứng viên (diverse beam search) → chấm điểm theo `W_NUMBER × number_recall + W_LENGTH × length_score - W_HALLUCINATION × number_hallucination` → chọn ứng viên cao nhất.

In [15]:
def generate_candidates(texts, k=5, batch_size=4):
    model.eval()
    all_cands = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i+batch_size], max_length=CONFIG['MAX_INPUT_LEN'],
                          truncation=True, padding=True, return_tensors='pt').to(model.device)
        with torch.no_grad():
            ids = model.generate(
                **batch,
                max_length=CONFIG['MAX_TARGET_LEN'],
                num_beams=k * 2,
                num_beam_groups=k,
                num_return_sequences=k,
                diversity_penalty=CONFIG['DIVERSITY_PENALTY'],
                no_repeat_ngram_size=CONFIG['NO_REPEAT_NGRAM'],
                length_penalty=CONFIG['LENGTH_PENALTY'],
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(ids, skip_special_tokens=True)
        for j in range(len(texts[i:i+batch_size])):
            all_cands.append(decoded[j*k:(j+1)*k])
    return all_cands

def length_score(src, pred, target=CONFIG['TARGET_RATIO']):
    return max(0.0, 1.0 - abs(length_ratio(src, pred) - target) / target)

def score_candidate(src, cand):
    return (
        CONFIG['W_NUMBER'] * number_recall(src, cand)
        + CONFIG['W_LENGTH'] * length_score(src, cand)
        - CONFIG['W_HALLUCINATION'] * number_hallucination(src, cand)
    )

def rerank_and_select(srcs, candidate_lists):
    picked, all_scores = [], []
    for src, cands in zip(srcs, candidate_lists):
        scores = [score_candidate(src, c) for c in cands]
        idx = int(np.argmax(scores))
        picked.append(cands[idx])
        all_scores.append(scores)
    return picked, all_scores

In [16]:
candidates = generate_candidates(test_src, k=CONFIG['NUM_CANDIDATES'], batch_size=4)
preds_rerank, scores = rerank_and_select(test_src, candidates)
print('Sample candidates:')
for c, s in zip(candidates[0], scores[0]):
    print(f'  [{s:.3f}] {c[:120]}...')

Sample candidates:
  [0.868] Tư lúc em gái qua đơi do viêm phôi năng, bà Phương , 70 tuôi, ghi chú lịch tiêm phòng hàng năm đê chông ôm. Cụ bà ở phườ...
  [0.868] Tư lúc em gái qua đơi do viêm phôi năng, bà Phương , 70 tuôi, ghi chú lịch tiêm phòng hàng năm đê chông ôm. Cụ bà ở phườ...
  [0.870] Tư lúc em gái qua đời do viêm phôi năng, bà PhươNg, 70 tuôi, ghi chú lịch tiêm phòng hàng năm đê chông ôm. Cụ bà ở phườn...
  [0.870] Tư lúc em gái qua đời do viêm phôi năng, bà PhươNg, 70 tuôi, ghi chú lịch tiêm phòng hàng năm đê chông ôm. Cụ bà ở phườn...
  [0.870] Tự lúc em gái qua đơi do viêm phôi năng, bà PhươNg, 70 tuôi, ghi chú lịch tiêm phòng hàng năm đê chông ôm. Cụ bà ở phườn...


## 9. So sánh SFT vs SFT+Rerank · export

In [17]:
metrics_rerank = evaluate_predictions(test_src, test_gold, preds_rerank, tag='SFT+Rerank')

import pandas as pd
df = pd.DataFrame([metrics_base, metrics_sft, metrics_rerank]).set_index('tag')
df

,rouge1,rouge2,rougeL,bertscore_f1,number_recall,number_hallucination,length_compliance,length_ratio_mean
tag,,,,,,,,
Checkpoint,28.99,16.66,21.34,83.97,5.74,2.22,0.00,0.0499
SFT,86.31,80.08,79.01,95.04,86.16,5.99,33.33,0.2217
SFT+Rerank,87.63,80.86,79.75,95.19,87.38,3.67,43.81,0.2219


In [18]:
rerank_records = [
    {'id': ds['test']['id'][i], 'source': test_src[i], 'gold': test_gold[i],
     'pred_base': preds_base[i], 'pred_sft': preds_sft[i], 'pred_rerank': preds_rerank[i],
     'candidates': candidates[i], 'cand_scores': scores[i]}
    for i in range(len(test_src))
]
final = {
    'model': CONFIG['MODEL_NAME'],
    'output_root': str(BASE_DIR),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'best_model_dir': str(BEST_DIR),
    'rl_model_dir': str(RL_MODEL_DIR),
    'metrics': {'checkpoint': metrics_base, 'sft': metrics_sft, 'sft_rerank': metrics_rerank},
    'config': CONFIG,
    'length_audit': length_audit,
    'records': rerank_records,
}
with open(RESULTS_DIR / 'final_results_stage2.json', 'w', encoding='utf-8') as f:
    json.dump(final, f, ensure_ascii=False, indent=2)
print('Saved final_results_stage2.json:', RESULTS_DIR / 'final_results_stage2.json')

Saved final_results_stage2.json: /content/drive/MyDrive/NLP_Final/Output stage 2 vit5 checkpoint/results/final_results_stage2.json
